<a href="https://colab.research.google.com/github/eyobedb/Multimodal-papaya-disease-classification-Leveraging-Computer-vision-and-NLP/blob/main/Multimodal_Resnet50_GPT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Concatenate, Input, Dropout
from tensorflow.keras.models import Model
from transformers import TFGPT2Model, GPT2Tokenizer

In [ ]:
# Paths to your image data
train_data_dir = 'path/to/your/train_data'
val_data_dir = 'path/to/your/val_data'

In [ ]:
# Number of classes
num_classes = 4

In [ ]:
# Image size for ResNet50
img_height, img_width = 224, 224

In [ ]:
# Preprocessing function for ResNet50
preprocess_input = tf.keras.applications.resnet50.preprocess_input


In [ ]:
# Load ResNet50 without the top layer
resnet_base = ResNet50(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))


In [ ]:
# Add new top layers for classification
x = resnet_base.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)  # Adding dropout for regularization
resnet_output = Dense(256, activation='relu')(x)

In [ ]:
# GPT-2 Model and Tokenizer
gpt2_model = TFGPT2Model.from_pretrained("gpt2")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [ ]:
def get_text_embeddings(texts):
    inputs = gpt2_tokenizer(texts, return_tensors="tf", padding=True, truncation=True, max_length=128)
    outputs = gpt2_model(inputs)
    return outputs.last_hidden_state[:, 0, :]  # Get the embedding for the [CLS] token


In [ ]:
# Inputs
image_input = Input(shape=(img_height, img_width, 3))
text_input = Input(shape=(128,), dtype=tf.int32)

In [ ]:
# Combine ResNet and GPT-2
text_embeddings = get_text_embeddings(text_input)
combined = Concatenate()([resnet_output, text_embeddings])
output = Dense(num_classes, activation='softmax')(combined)

In [ ]:
# Final model
model = Model(inputs=[image_input, text_input], outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
# Prepare image data generator
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_input)
val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_input)


In [ ]:
# Create generators
train_image_gen = train_datagen.flow_from_directory(train_data_dir, target_size=(img_height, img_width), class_mode='categorical', batch_size=32)
val_image_gen = val_datagen.flow_from_directory(val_data_dir, target_size=(img_height, img_width), class_mode='categorical', batch_size=32)


In [ ]:
# Function to yield image and text data
def data_generator(image_gen):
    while True:
        image_batch, label_batch = image_gen.next()
        text_batch = [get_text_embeddings(["Papaya disease description."] * len(label_batch))]  # Example text input
        yield [image_batch, text_batch], label_batch

In [ ]:
# Train the model
train_data_gen = data_generator(train_image_gen)
val_data_gen = data_generator(val_image_gen)


In [ ]:
model.fit(train_data_gen, steps_per_epoch=len(train_image_gen), validation_data=val_data_gen, validation_steps=len(val_image_gen), epochs=10)


In [ ]:
# Save the model
model.save('papaya_disease_multimodal_model.h5')